## Problem Statement

### **Business Context**

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### **Objective**

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### **Data Description**

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [1]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.1.85 --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 34.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 259.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 248.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 278.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.5 which is incompatible.


In [1]:
# For installing the libraries & downloading models from HF Hub
!pip install huggingface_hub==0.35.3 pandas==2.2.2 tiktoken==0.12.0 pymupdf==1.26.5 langchain==0.3.27 langchain-community==0.3.31 chromadb==1.1.1 sentence-transformers==5.1.1 numpy==2.3.3 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 109.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 117.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 126.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 486.6/486.6 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 131.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 106.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [1]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

## Question Answering using LLM

#### **Downloading and Loading the model**

In [2]:
# define the Hugging Face model repository and model filename for the Mistral-7B-Instruct-v0.2 GGUF model.
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"

In [3]:
# Using hf_hub_download to download a model from the Hugging Face model hub
# The repo_id parameter specifies the model name or path in the Hugging Face repository
# The filename parameter specifies the name of the file to download
model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


mistral-7b-instruct-v0.2.Q6_K.gguf:   0%|          | 0.00/5.94G [00:00<?, ?B/s]

In [4]:
# Initialize the model
llm = Llama(
    model_path=model_path, # path to GGUF model
    n_threads=2,  # CPU cores
    n_batch=512,  # batch size for processing
    n_ctx=4096,  # Context window
)

AVX = 1 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


#### **Utility function `response`**

## Model Parameters Used

The following parameters were used for response generation:

- **Temperature**: Controls randomness in responses
- **max_tokens**: Limits response length
- **top_p**: Controls nucleus sampling
- **top-k**: Restricts token selection to top probable tokens

In [5]:
# function to generate the response from the LLM
def response(query,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )
    # Extract and return the response text
    return model_output['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [85]:
query_1 = "What is the protocol for managing sepsis in a critical care unit?"

answer = response(query_1)
print (answer)

Llama.generate: prefix-match hit




Sepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit. The following are general steps for managing sepsis in a critical care unit:

1. Early recognition and suspicion: Septic patients may present with non-specific symptoms such as fever, chills, tachycardia, tachypnea, altered mental status, or lactic acidosis. It is essential to have a high index of suspicion for sepsis in patients with known or suspected infections.
2. Initial assessment and res


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [86]:
query_2 = "What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

answer = response(query_2)
print (answer)

Llama.generate: prefix-match hit




Appendicitis is a medical condition characterized by inflammation of the appendix, a small tube-shaped organ located in the lower right side of the abdomen. The symptoms of appendicitis can vary from person to person, but some common signs include:

1. Abdominal pain: The pain typically starts as a mild discomfort around the navel or in the lower right abdomen. It may then move to the right lower quadrant and become more severe over time. The pain is often described as sharp, crampy, or constant.
2. Loss of appetite: People with


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [87]:
query_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

answer = response(query_3)
print (answer)

Llama.generate: prefix-match hit




Sudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects the hair follicles. It can result in round or oval bald patches on the scalp, but it can also occur on other parts of the body such as the beard area, eyebrows, and eyelashes.

The exact cause of alopecia areata is not known, but it's believed to be related to a problem with the immune system. Some possible triggers for this condition include stress, genetics, viral infections, and certain medications.


### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [88]:
query_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"

answer = response(query_4)
print (answer)

Llama.generate: prefix-match hit




There is no one-size-fits-all answer to this question as the specific treatment recommendations for a person with a brain injury depend on various factors such as the severity and location of the injury, the extent of functional impairment, and the individual's overall health and medical history. However, I can provide some general information about common treatments and interventions used to manage brain injuries.

1. Acute care: In the immediate aftermath of a brain injury, the priority is to ensure the person's safety and stabilize their condition. This may involve hospitalization, monitoring vital signs, managing


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [89]:
query_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"

answer = response(query_5)
print (answer)

Llama.generate: prefix-match hit




First and foremost, if you suspect that someone has fractured their leg while hiking, it's essential to ensure their safety and prevent further injury. Here are some necessary precautions:

1. Keep the person calm and still: Encourage them to remain as still as possible to minimize pain and prevent worsening the injury.
2. Assess the situation: Check for any signs of shock, such as pale skin, rapid heartbeat, or shallow breathing. If you notice these symptoms, seek medical help immediately.
3. Immobilize the leg: Use a splint, sl


### **Observations: Question Answering Using Only LLM**

*   The **Mistral-7B-Instruct** model generated **coherent and relevant but generic responses** for all the queries, including sepsis management, appendicitis, hair loss, brain injury and leg fracture.
*   However, since the model used pre-trained knowledge for response generation, the responses are **not grounded in external medical documents** which can lead to **possible hallucinations** and **unverified information**.
*   The **responses were truncated** due to the default **max_tokens** limit of **128**.





## Question Answering using LLM with Prompt Engineering

In [7]:
# system message describing the assistant's role
system_prompt = """
You are an AI assistant who has expertise in the medical domain.
You are expected to provide medically accurate, concise, and clear answers based on standard clinical guidelines and trusted medical knowledge.

Your responses should be acceptable to healthcare professionals.

If the answer is not known or insufficient information is available, respond with:
"I don't know the answer to this question."
"""


### Query 1: What is the protocol for managing sepsis in a critical care unit?

**1) Parameter Being Tested: temperature**



In [12]:
input_1 = system_prompt + "\n" + "What is the protocol for managing sepsis in a critical care unit?"

# Combination 1 (temperature = 0.0)
answer = response (input_1)
print (answer)

Llama.generate: prefix-match hit




The protocol for managing sepsis in a critical care unit involves a systematic approach that includes early recognition, rapid resuscitation, source control, and ongoing supportive care. Here are the key steps:

1. Recognition: Identify patients who may have sepsis based on clinical signs and symptoms such as fever or hypothermia, tachycardia or bradycardia, respiratory distress, altered mental status, and lactic acidosis. Use the Sequential [Sepsis-related] Organ Failure Assessment (SOFA) score to help identify patients with se


In [13]:
input_1 = system_prompt + "\n" + "What is the protocol for managing sepsis in a critical care unit?"

# Combination 2 (temperature = 0.2)
answer = response(input_1, temperature=0.2)
print (answer)

Llama.generate: prefix-match hit




The protocol for managing sepsis in a critical care unit involves a systematic approach that includes early recognition, rapid response, and appropriate treatment. Here are the key steps:

1. Recognition: Identify patients who may have sepsis based on clinical signs and symptoms such as fever or hypothermia, tachycardia or bradycardia, respiratory distress, altered mental status, and lactic acidosis. Use the Sequential [Sepsis-related] Organ Failure Assessment (SOFA) score to help identify patients at risk for sepsis and sept


In [14]:
input_1 = system_prompt + "\n" + "What is the protocol for managing sepsis in a critical care unit?"

# Combination 3 (temperature = 0.4)
answer = response(input_1, temperature=0.4)
print (answer)

Llama.generate: prefix-match hit




The protocol for managing sepsis in a critical care unit involves the following steps:
1. Early recognition and diagnosis: Recognize sepsis early based on clinical suspicion, vital signs, and laboratory findings such as lactate level, white blood cell count, and C-reactive protein level.
2. Immediate fluid resuscitation: Administer intravenous fluids to maintain adequate tissue perfusion and prevent hypotension. The goal is to achieve a mean arterial pressure (MAP) of 65 mmHg or higher.
3. Antibiotic therapy: Start


In [15]:
input_1 = system_prompt + "\n" + "What is the protocol for managing sepsis in a critical care unit?"

# Combination 4 (temperature = 0.6)
answer = response(input_1, temperature=0.6)
print (answer)

Llama.generate: prefix-match hit




The protocol for managing sepsis in a critical care unit involves a systematic approach called the Surviving Sepsis Campaign Bundle. This bundle includes the following interventions:

1. Early recognition and evaluation: Identify patients with suspected infection and assess for sepsis using the Sequential [Sepsis-related] Organ Failure Assessment (SOFA) score or other recognized scoring systems.
2. Source control: Address the source of infection as soon as possible, which may involve surgical intervention, drainage, or debridement.
3. Fluid resuscitation: Administer


In [16]:
input_1 = system_prompt + "\n" + "What is the protocol for managing sepsis in a critical care unit?"

# Combination 5 (temperature = 0.8)
answer = response(input_1, temperature=0.8)
print (answer)

Llama.generate: prefix-match hit




Sepsis is a life-threatening condition that can arise from an infection. The management of sepsis in a critical care unit involves a multidisciplinary approach, focusing on early recognition, prompt initiation of appropriate antibiotic therapy, and supportive measures to maintain organ function. Here are the general steps for managing sepsis in a critical care unit:

1. Early recognition: Recognize sepsis early based on clinical suspicion, vital signs, and laboratory findings such as elevated heart rate, respiratory rate, temperature, and white blood cell count. Use tools like Sequ


**Observations:**
*   As the temperature increased from 0.2 to 0.8, the responses became more diverse and explanatory.
*   Lower to moderate temperature values gave more clear, concise and clinically relevant answers where as higher temperature values resulted in a lot of additional supporting information.
*   At **temperature 0.0**, the response was **very focused and well presented** and prioritized sepsis management**.
*   At **temperature 0.8**, the response was **very descriptive with additional information** before discussing sepsis management.



### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

**2) Parameter Being Tested: max_tokens**



In [11]:
input_2 = system_prompt + "\n" + "What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

# Combination 1 (max_tokens = 128)
answer = response (input_2)
print (answer)

Llama.generate: prefix-match hit




Appendicitis is a medical condition characterized by inflammation of the appendix, a small pouch that extends from the cecum (the beginning of the large intestine). The common symptoms of appendicitis include:
1. Sudden onset of pain in the lower right abdomen, which may be mild at first but becomes sharp and severe over time.
2. Loss of appetite and feeling sick to your stomach (nausea) with or without vomiting.
3. Fever, usually above 37.5°C (99.5°F).
4


In [18]:
input_2 = system_prompt + "\n" + "What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

# Combination 2 (max_tokens = 256)
answer = response(input_2, max_tokens=256)
print (answer)

Llama.generate: prefix-match hit




Appendicitis is a medical condition characterized by inflammation of the appendix, a small pouch that extends from the cecum (the beginning of the large intestine). The common symptoms of appendicitis include:
1. Sudden onset of pain in the lower right abdomen, often described as sharp or crampy and worsening with movement or pressure.
2. Loss of appetite and nausea, which may lead to vomiting.
3. Fever, usually above 37.5°C (99.5°F).
4. Constipation or diarrhea.
5. Abdominal swelling and tenderness.
6. Inability to pass gas or have a bowel movement.

Appendicitis cannot be cured via medicine alone, as the inflammation can progress to rupture, leading to peritonitis, a potentially life-threatening condition. If diagnosed early, appendicitis is typically treated with an appendectomy, which involves surgically removing the appendix. In some cases, a laparoscopic appendectomy may be performed, which uses small incisions and a camera to guide


In [19]:
input_2 = system_prompt + "\n" + "What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

# Combination 3 (max_tokens = 512)
answer = response(input_2, max_tokens=512)
print (answer)

Llama.generate: prefix-match hit




Appendicitis is a medical condition characterized by inflammation of the appendix, a small pouch that extends from the cecum (the beginning of the large intestine). The common symptoms of appendicitis include:
1. Sudden onset of pain in the lower right abdomen, often described as sharp or crampy and worsening with movement or pressure.
2. Loss of appetite and nausea, which may lead to vomiting.
3. Fever, usually above 37.5°C (99.5°F).
4. Constipation or diarrhea.
5. Abdominal swelling and tenderness.
6. Inability to pass gas or have a bowel movement.

Appendicitis cannot be cured via medicine alone, as the inflammation can progress to rupture, leading to peritonitis, a potentially life-threatening condition. If diagnosed early, appendicitis is typically treated with an appendectomy, which involves surgically removing the appendix. In some cases, a laparoscopic appendectomy may be performed, which uses small incisions and a camera to guide the surgery. However, if the appendix has rup

In [20]:
input_2 = system_prompt + "\n" + "What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

# Combination 4 (max_tokens = 1024)
answer = response(input_2, max_tokens=1024)
print (answer)

Llama.generate: prefix-match hit




Appendicitis is a medical condition characterized by inflammation of the appendix, a small pouch that extends from the cecum (the beginning of the large intestine). The common symptoms of appendicitis include:
1. Sudden onset of pain in the lower right abdomen, often described as sharp or crampy and worsening with movement or pressure.
2. Loss of appetite and nausea, which may lead to vomiting.
3. Fever, usually above 37.5°C (99.5°F).
4. Constipation or diarrhea.
5. Abdominal swelling and tenderness.
6. Inability to pass gas or have a bowel movement.

Appendicitis cannot be cured via medicine alone, as the inflammation can progress to rupture, leading to peritonitis, a potentially life-threatening condition. If diagnosed early, appendicitis is typically treated with an appendectomy, which involves surgically removing the appendix. In some cases, a laparoscopic appendectomy may be performed, which uses small incisions and a camera to guide the surgery. However, if the appendix has rup

**Observations:**
*   **Increasing max_tokens** from 128 to 512 had a **significant improvement on completeness** of response.
*   **Lower max_tokens** value of 128 gave **truncated response** resulting in **incomplete information**.
*   **Higher max_tokens** value of 512 gave **complete** and **medically coherent response**.
*   The responses at max_token value of 512 and 1024 were almost identical suggesting that increasing the token limit beyond 512 did not have any impact on the response.



### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

**3) Parameter Being Tested: top_p**



In [22]:
input_3 = system_prompt + "\n" + "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

# Combination 1 (top_p = 0.70)
answer = response(input_3, top_p = 0.70)
print (answer)

Llama.generate: prefix-match hit




Possible causes of sudden patchy hair loss, also known as alopecia areata, include:
1. Autoimmune disorders: The immune system mistakenly attacks hair follicles, leading to hair loss.
2. Stress: Emotional or physical stress can trigger hair loss.
3. Genetics: A family history of alopecia areata increases the risk.
4. Viral infections: Certain viruses may cause hair loss.
5. Medications: Some drugs, such as those used for cancer treatment, can lead to hair loss.

Effect


In [23]:
input_3 = system_prompt + "\n" + "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

# Combination 2 (top_p = 0.80)
answer = response(input_3, top_p = 0.80)
print (answer)

Llama.generate: prefix-match hit




Possible causes of sudden patchy hair loss, also known as alopecia areata, include:
1. Autoimmune disorders: The immune system mistakenly attacks hair follicles, leading to hair loss.
2. Stress: Emotional or physical stress can trigger hair loss.
3. Genetics: A family history of alopecia areata increases the risk.
4. Viral infections: Certain viruses may cause hair loss.
5. Medications: Some drugs, such as those used for cancer treatment, can lead to hair loss.

Effect


In [24]:
input_3 = system_prompt + "\n" + "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

# Combination 3 (top_p = 0.90)
answer = response(input_3, top_p = 0.90)
print (answer)

Llama.generate: prefix-match hit




Possible causes of sudden patchy hair loss, also known as alopecia areata, include:
1. Autoimmune disorders: The immune system mistakenly attacks hair follicles, leading to hair loss.
2. Stress: Emotional or physical stress can trigger hair loss.
3. Genetics: A family history of alopecia areata increases the risk.
4. Viral infections: Certain viruses may cause hair loss.
5. Medications: Some drugs, such as those used for cancer treatment, can lead to hair loss.

Effect


In [25]:
input_3 = system_prompt + "\n" + "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

# Combination 4 (top_p = 0.95)
answer = response(input_3, top_p = 0.95)
print (answer)

Llama.generate: prefix-match hit




Possible causes of sudden patchy hair loss, also known as alopecia areata, include:
1. Autoimmune disorders: The immune system mistakenly attacks hair follicles, leading to hair loss.
2. Stress: Emotional or physical stress can trigger hair loss.
3. Genetics: A family history of alopecia areata increases the risk.
4. Viral infections: Certain viruses may cause hair loss.
5. Medications: Some drugs, such as those used for cancer treatment, can lead to hair loss.

Effect


**Observations:**
*   Changing the **top_p value from 0.70 to 0.95** resulted in  **robust** responses across all the top_p values.



### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

**4) Parameter Being Tested: top_k**



In [26]:
input_4 = system_prompt + "\n" + "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"

# Combination 1 (top_k = 20)
answer = response(input_4, top_k = 20)
print (answer)

Llama.generate: prefix-match hit




The treatment for a person with a brain injury depends on the severity and location of the injury. Here are some common treatments:

1. Emergency care: For severe brain injuries, immediate medical attention is required to prevent further damage. This may include surgery to remove hematomas or decompressing skull fractures.
2. Medications: Depending on the symptoms, medications may be prescribed to manage conditions such as seizures, pain, or swelling in the brain.
3. Rehabilitation: Rehabilitation is an essential part of treatment for brain injuries. It may include physical therapy,


In [27]:
input_4 = system_prompt + "\n" + "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"

# Combination 2 (top_k = 30)
answer = response(input_4, top_k = 30)
print (answer)

Llama.generate: prefix-match hit




The treatment for a person with a brain injury depends on the severity and location of the injury. Here are some common treatments:

1. Emergency care: For severe brain injuries, immediate medical attention is required to prevent further damage. This may include surgery to remove hematomas or decompressing skull fractures.
2. Medications: Depending on the symptoms, medications may be prescribed to manage conditions such as seizures, pain, or swelling in the brain.
3. Rehabilitation: Rehabilitation is an essential part of treatment for brain injuries. It may include physical therapy,


In [28]:
input_4 = system_prompt + "\n" + "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"

# Combination 3 (top_k = 50)
answer = response(input_4, top_k = 50)
print (answer)

Llama.generate: prefix-match hit




The treatment for a person with a brain injury depends on the severity and location of the injury. Here are some common treatments:

1. Emergency care: For severe brain injuries, immediate medical attention is required to prevent further damage. This may include surgery to remove hematomas or decompressing skull fractures.
2. Medications: Depending on the symptoms, medications may be prescribed to manage conditions such as seizures, pain, or swelling in the brain.
3. Rehabilitation: Rehabilitation is an essential part of treatment for brain injuries. It may include physical therapy,


In [29]:
input_4 = system_prompt + "\n" + "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"

# Combination 4 (top_k = 70)
answer = response(input_4, top_k = 70)
print (answer)

Llama.generate: prefix-match hit




The treatment for a person with a brain injury depends on the severity and location of the injury. Here are some common treatments:

1. Emergency care: For severe brain injuries, immediate medical attention is required to prevent further damage. This may include surgery to remove hematomas or decompressing skull fractures.
2. Medications: Depending on the symptoms, medications may be prescribed to manage conditions such as seizures, pain, or swelling in the brain.
3. Rehabilitation: Rehabilitation is an essential part of treatment for brain injuries. It may include physical therapy,


**Observations:**
*   Changing the **top_k value from 20 to 70** resulted in **minimal variation in responses**.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

**5) Best performing parameters settings**



In [30]:
input_5 = system_prompt + "\n" + "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"

# Best parameter combination

answer = response(
    input_5,
    temperature=0.2,
    max_tokens=512,
    top_p=0.95,
    top_k=50
)
print (answer)

Llama.generate: prefix-match hit




A leg fracture, specifically a tibia or fibula fracture, can occur due to falls or traumatic events such as hiking accidents. Here are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip:

1. Assess the severity of the injury: Check for signs of open wounds, swelling, deformity, numbness, or inability to move the leg. If there are any signs of severe bleeding, open wounds, or inability to bear weight on the leg, seek emergency medical help immediately.
2. Immobilize the leg: Use a splint, sling, or a makeshift immobilizer to prevent further movement and reduce pain. Be sure not to apply too much pressure on the injury site as it may cause more damage.
3. Provide comfort and support: Keep the injured person warm and comfortable. Encourage them to rest and elevate their leg to reduce swelling and promote healing.
4. Administer first aid: Clean the wound, if present, with water and mild soap. Apply a sterile dressing and cover it w

**Observations:**

*   The response was detailed without being overly descriptive.
*   Lower temperature value gave focused response without randomness.
*   Higher token value resulted in complete response.
*   top_p and top_k maintained balanced response diversity and coherence.

**Final Observations on using Prompt Engineering:**
*   After introducing system prompts and response guidelines, the quality of the responses for all the queries improved and the model gave **less generic, more focused, concise** and **medically relevant** responses.
    *   **Example:**
          *   **Without Prompt Engineering:** Sepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit.
          *   **With Prompt Engineering:** The protocol for managing sepsis in a critical care unit involves early recognition and prompt initiation of appropriate therapy.
*   **Lower temperatures** resulted in **more focussed and contextually correct responses**.
*   **Increasing max_tokens** was beneficial for **large queries** as it resulted in **complete responses**.




## **Data Preparation for RAG**

### **Loading the Data**

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
merck_manual_pdf_path = "/content/drive/MyDrive/PythonCourse/Natural_Language_Processing_with_Generative_AI/Project/medical_diagnosis_manual.pdf"

In [7]:
pdf_loader = PyMuPDFLoader(merck_manual_pdf_path)

In [8]:
merck_manual = pdf_loader.load()

**Observations:**
*   The Merck manual was successfully loaded using PyMuPDFLoader.





### **Data Overview**

#### Checking the first 5 pages

In [ ]:
for i in range(5):
    print(f"Page Number : {i+1}",end="\n")
    print(merck_manual[i].page_content,end="\n")

Page Number : 1
jamalsalma1@gmail.com
H27MS6WAL3
This file is meant for personal use by jamalsalma1@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 2
jamalsalma1@gmail.com
H27MS6WAL3
This file is meant for personal use by jamalsalma1@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    ...............................................................................................................................

#### Checking the number of pages

In [ ]:
len(merck_manual)

4114

**Observations:**
*   Sanity checks were done on the data confirming extraction of pages and verifying the pages from the manual successfully.



### **Data Chunking**

**Text Chunking Strategy**

The dataset was split into smaller chunks before embedding generation.

Parameters used:
- **Chunk Size**: 512
- **Chunk Overlap**: 20

In [9]:
# Create an instance of text splitter
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap= 20
)

In [10]:
document_chunks = pdf_loader.load_and_split(text_splitter)

In [ ]:
# get the number of chunks
len(document_chunks)

8469

In [ ]:
# get the first chunk
document_chunks[0].page_content

'jamalsalma1@gmail.com\nH27MS6WAL3\nThis file is meant for personal use by jamalsalma1@gmail.com only.\nSharing or publishing the contents in part or full is liable for legal action.'

In [ ]:
# get the last second chunk
document_chunks[-2].page_content

'Y\nYaws 1266-1267\nforest 1379\nY chromosome 3373 (see also Genetic)\nabnormalities of 3005\nYeast infection (see also Fungal infection)\nvaginal 2542, 2544, 2545\nYellow fever 1400, 1429, 1437\nhepatic inflammation in 248\nvaccine against 1172, 1437, 3441\nYellow nail syndrome 732, 1995\npleural effusion in 1997\nYellow skin (see Jaundice)\nYersinia infection 1167, 1256-1257\nY. enterocolitica infection 147\nY. pestis infection 1924\nYew poisoning 3338\nYips 1762\nYo, antibodies to 1056\nYolk sac tumor 2476\nThe Merck Manual of Diagnosis & Therapy, 19th Edition\nY\n4103\njamalsalma1@gmail.com\nH27MS6WAL3\nThis file is meant for personal use by jamalsalma1@gmail.com only.\nSharing or publishing the contents in part or full is liable for legal action.'

In [ ]:
# get the last chunk
document_chunks[-1].page_content

"Z\nZafirlukast 1879\nZalcitabine 1451\nin children 2854\nZaleplon 1709\nZanamivir 1407\nin influenza 1407, 1929\nZAP-70 (zeta-associated protein 70) deficiency 1092, 1108\nZavanelli maneuver 2680\nZellweger syndrome 2383, 3023\nZenker's diverticulum 125\nZidovudine 1451, 1453\nin children 2854\nZileuton 1881\nin asthma 1880\nZinc 49, 55, 3431-3432\nin common cold 1405\ndeficiency of 11, 49, 55\nin dermatophytoses 705\npoisoning with 3328, 3353\nrecommended dietary allowances for 50\nreference values for 3499\ntoxicity of 49, 55\ncopper deficiency and 49\nin Wilson's disease 52\nZinc oxide 2233\ngelatin formulation of 646, 672\nZinc pyrithione 647\nZinc shakes 55\nZipper injury 3239, 3240\nZiprasidone\nin agitation 1492\nin bipolar disorder 3059\npoisoning with 3347\nin schizophrenia 1566\nZoledronate 359, 361, 848\nZollinger-Ellison syndrome 95, 199, 200-201, 910\nmastocytosis vs 1125\nMenetrier's disease vs 132\npeptic ulcer disease vs 134\nZolmitriptan 1721\nZolpidem 1709, 3103\nZon

**Observations:**
*   The manual was divided into smaller chunks to improve the retieval efficieny and embedding process.
*   The inspection of the few initial and final chunks suggested that all the chunks contained clinical terminologies.



### **Embedding**

In [11]:
# loading the GTE-large embedding model
embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')

/tmp/ipykernel_3243/4198310515.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')


modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [12]:
# creating embedding vectors for the first two document chunks
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

In [13]:
# checking the embedding vector dimensions
print("Dimension of the embedding vector ",len(embedding_1))
len(embedding_1)==len(embedding_2)

Dimension of the embedding vector  1024


True

**Observations:**
*   The embedding vector dimensions were consistent.




### **Vector Database**

In [14]:
out_dir = 'merck_manual_db'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

In [15]:
# Create Embeddings for each of the Text Chunk using Chroma database
print("Starting embedding process...")

vectorstore = Chroma.from_documents(
    document_chunks,
    embedding_model,
    persist_directory=out_dir
)
print("Embedding completed and saved!")

Starting embedding process...
Embedding completed and saved!


In [16]:
# saving the embeddings
vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)

/tmp/ipykernel_3243/2756559696.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)


In [ ]:
vectorstore.embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='thenlper/gte-large', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

### **Retriever**

**Retriever Configuration**

- **Search Type:** similarity
- **Top-k Chunks Retrieved:** 3

In [17]:
# instantiating retriever databases using similarity search
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 2}
)

In [18]:
# retrieving contextually related document chunks
rel_docs = retriever.get_relevant_documents("What is the protocol for managing sepsis in a critical care unit?")
print (rel_docs)

[Document(metadata={'source': '/content/drive/MyDrive/PythonCourse/Natural_Language_Processing_with_Generative_AI/Project/medical_diagnosis_manual.pdf', 'moddate': '2026-05-12T07:22:11+00:00', 'format': 'PDF 1.7', 'modDate': 'D:20260512072211Z', 'page': 2400, 'file_path': '/content/drive/MyDrive/PythonCourse/Natural_Language_Processing_with_Generative_AI/Project/medical_diagnosis_manual.pdf', 'keywords': '', 'author': '', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creationDate': 'D:20120615054440Z', 'trapped': '', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'creator': 'Atop CHM to PDF Converter', 'creationdate': '2012-06-15T05:44:40+00:00', 'total_pages': 4114, 'subject': ''}, page_content="16 - Critical Care Medicine\nChapter 222. Approach to the Critically Ill Patient\nIntroduction\nCritical care medicine specializes in caring for the most seriously ill patients. These patients are best\ntreated in an ICU staffed by experienced personnel. Some 

/tmp/ipykernel_3243/3394886873.py:1: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  rel_docs = retriever.get_relevant_documents("What is the protocol for managing sepsis in a critical care unit?")


**Observations:**
*   The retriever successfully fetched the clinically relveant documents related to sepsis management.




### **System and User Prompt Template**

In [24]:
# system message describing the assistant's role
qna_system_message = """
You are an AI medical assistant with expertise in clinical and healthcare domains.

Your task is to answer medical questions using ONLY the information provided in the context extracted from the Merck Manual.

This context will begin with the token: ###Context.
Relevant excerpts retrieved from the Merck Manual.

User questions will begin with the token: ###Question.
The medical question to be answered.

Guidelines:
1. Provide medically accurate, concise, and clear answers.
2. Please answer only using the context provided in the input.
3. Do not add external medical knowledge or assumptions.
4. Do not mention anything about the context in your final answer.
5. If the answer is not found in the context, respond "I don't know".
"""


In [25]:
# user message template including context and the question
qna_user_message_template = """
###Context
Here are some documents that are relevant to the question mentioned below.
{context}

###Question
{question}
"""

### **Response Function**

In [21]:
def generate_rag_response(user_input,k=3,max_tokens=512,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## **Question Answering using RAG**

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [26]:
user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"
answer = generate_rag_response (user_input_1)
print (answer)

###Answer
The protocol for managing sepsis in a critical care unit includes administering supportive care, monitoring vital signs and physiologic parameters, performing routine blood tests, providing prompt empiric antibiotic therapy, draining abscesses or excising necrotic tissues, normalizing blood glucose levels, considering corticosteroid therapy, and assessing the need for emerging therapies such as cooling for hyperthermia and early treatment of renal failure. The Merck Manual suggests a regimen of gentamicin or tobramycin plus a 3rd-generation cephalosporin for initial antibiotic therapy, with vancomycin added if resistant staphylococci or enterococci are suspected. Antibiotics should be continued for at least 5 days after shock resolves and evidence of infection subsides. Normalization of blood glucose improves outcome in critically ill patients, and a continuous IV insulin infusion is used to maintain glucose levels between 80 to 110 mg/dL. Corticosteroid therapy with replacem

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [27]:
user_input_2 = "What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
answer = generate_rag_response (user_input_2)
print (answer)

Llama.generate: prefix-match hit


###Answer
The common symptoms of appendicitis include abdominal pain, anorexia, and abdominal tenderness. Appendicitis cannot be cured via medicine alone; the standard treatment is surgical removal of the appendix (appendectomy).


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [28]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
answer = generate_rag_response (user_input_3)
print (answer)

Llama.generate: prefix-match hit


Based on the context provided, the condition being described is alopecia areata. The treatment options for this condition include topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA). The cause of alopecia areata is believed to be an autoimmune disorder affecting genetically susceptible people exposed to unclear environmental triggers. It's important to note that the context does not provide information on the effectiveness or success rates of these treatments, only that they are options for addressing alopecia areata.


### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [29]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
answer = generate_rag_response (user_input_4)
print (answer)

Llama.generate: prefix-match hit


I. Supportive care: Preventing systemic complications due to immobilization (eg, pneumonia, UTI, thromboembolic disease), providing good nutrition, and preventing pressure ulcers are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function.

II. Initial treatment: Ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure is the initial treatment for a person with traumatic brain injury (TBI). Surgery may be needed in patients with more severe injury to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas.

III. Medications: Drugs are given to prevent seizures if head injury causes significant structural injury (eg, large contusions or hematomas, brain laceration, depressed skull fracture) or a Glasgow Coma Scale (GCS) score of < 10. These drugs reduce the risk of seiz

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [30]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
answer = generate_rag_response (user_input_5)
print (answer)

Llama.generate: prefix-match hit


Based on the context provided, here's the answer:

The person with a fractured leg should follow these steps:
1. Rest and elevate the injured limb above heart level for the first 2 days to minimize swelling.
2. Apply ice after 48 hours to relieve pain and speed healing.
3. Immobilize the injured limb using either a cast or splint, depending on the severity of the injury and the length of immobilization required.
   - For fractures that require weeks of immobilization, use a cast.
   - For injuries that only need several days of immobilization, use a splint.
4. Keep the cast dry and never put any objects inside it.
5. Inspect the skin around the cast daily for red or sore areas and apply lotion to these areas.
6. Pad any rough edges with soft material to prevent irritation.
7. Seek medical care immediately if an odor emanates from within the cast, a fever develops, or there is a risk of infection.
8. If possible, resume active motion as soon as possible for rapidly healing injuries to m

**Observations:**
*   Compared to standalone LLM, RAG pipeline generated **less generic** and **more clinically specific and context grounded responses**.
*   The responses were **medically reliable** included **clinical terminology**, and were **treatment-oriented**.

### **Fine-tuning: RAG Retriever and LLM Parameters**

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [60]:
user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"

# Combination 1 (k = 2, temperature = 0.2, top_p = 0.8)
answer = generate_rag_response(
    user_input_1,
    k=2,
    temperature = 0.2,
    top_p = 0.8
)
print (answer)

Llama.generate: prefix-match hit


###Answer
The protocol involves administering antibiotics such as gentamicin or tobramycin along with a third-generation cephalosporin, ceftazidime, or a fluoroquinolone based on the suspected source and organisms. Vancomycin should be added if resistant staphylococci or enterococci are suspected. Abscesses must be drained, and necrotic tissues need to be surgically excised. Normalization of blood glucose is important for improving outcomes in critically ill patients using a continuous IV insulin infusion titrated to maintain glucose levels between 80 to 110 mg/dL. Antibiotics should be continued for at least 5 days after shock resolves and evidence of infection subsides.


In [66]:
user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"

# Combination 2 (k = 3, temperature = 0.3, top_p = 0.85)
answer = generate_rag_response(
    user_input_1,
    k=3,
    temperature = 0.3,
    top_p = 0.85
)
print (answer)

Llama.generate: prefix-match hit


###Answer
The protocol for managing sepsis in a critical care unit includes administering antibiotics such as gentamicin or tobramycin, a 3rd-generation cephalosporin, ceftazidime, or ciprofloxacin based on the suspected source and sensitivity patterns. Vancomycin should be added if resistant staphylococci or enterococci are suspected. Abscesses must be drained, and necrotic tissues must be surgically excised. Normalization of blood glucose is important with a continuous IV insulin infusion to maintain glucose levels between 80 to 110 mg/dL. Corticosteroid therapy with hydrocortisone and fludrocortisone may also be beneficial during hemodynamic instability and for 3 days thereafter. Activated protein C (drotrecogin alfa) is an emerging therapy for severe sepsis if begun early, but it carries a risk of bleeding. Other emerging therapies include cooling for hyperthermia and early treatment of renal failure with continuous venovenous hemofiltration.


In [75]:
user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"

# Combination 3 (k = 4, temperature = 0.5, top_p = 0.9)
answer = generate_rag_response(
    user_input_1,
    k=4,
    temperature = 0.5,
    top_p = 0.9
)
print (answer)

Llama.generate: prefix-match hit


Based on the context provided, the protocol for managing sepsis in a critical care unit includes:
1. Aggressive fluid resuscitation
2. Antibiotics (gentamicin or tobramycin 5.1 mg/kg IV once/day plus a 3rd-generation cephalosporin, ceftazidime 2 g IV q 8 h for unknown cause; ceftazidime plus a fluoroquinolone if Pseudomonas is suspected)
3. Surgical excision of infected or necrotic tissues and drainage of pus
4. Supportive care
5. Normalization of blood glucose with continuous IV insulin infusion (crystalline zinc 1 to 4 U/h)
6. Corticosteroid therapy with replacement doses (hydrocortisone 50 mg IV q 6 h or 100 mg q 8 h plus fludrocortisone 50 μg po once/day during hemodynamic instability and for 3 days thereafter)
7. Activated protein C (drotrecogin alfa) if begun early and the patient has a significant risk of death as defined by an APACHE II score of > 25
8. Other emerging therapies like cooling for hyperthermia and early treatment of renal failure with continuous venovenous hemofil

In [76]:
user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"

# Combination 4 (k = 5, temperature = 0.7, top_p = 0.95)
answer = generate_rag_response(
    user_input_1,
    k=5,
    temperature = 0.7,
    top_p = 0.95
)
print (answer)

Llama.generate: prefix-match hit


###Answer
The protocol for managing sepsis in a critical care unit includes aggressive fluid resuscitation, antibiotics, surgical excision of infected or necrotic tissues and drainage of pus, supportive care, and sometimes intensive control of blood glucose and administration of corticosteroids and activated protein C. Septic shock is severe sepsis with organ hypoperfusion and hypotension that are poorly responsive to initial fluid resuscitation. Antibiotic selection requires an educated guess based on the suspected source, clinical setting, knowledge or suspicion of causative organisms and of sensitivity patterns common to that specific inpatient unit, and previous culture results. A regimen for septic shock of unknown cause is gentamicin or tobramycin 5.1 mg/kg IV once/day plus a 3rd-generation cephalosporin (cefotaxime 2 g q 6 to 8 h or ceftriaxone 2 g once/day) or ceftazidime 2 g IV q 8 h, with vancomycin added if resistant staphylococci or enterococci are suspected. Abscesses must

In [77]:
user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"

# Combination 5 (k = 5, temperature = 0.4, top_p = 0.92)
answer = generate_rag_response(
    user_input_1,
    k=5,
    temperature = 0.4,
    top_p = 0.92
)
print (answer)

Llama.generate: prefix-match hit


Based on the context provided, the following answer can be given:
The protocol for managing sepsis in a critical care unit includes aggressive fluid resuscitation, antibiotics, surgical excision of infected or necrotic tissues and drainage of pus, supportive care, and sometimes intensive control of blood glucose and administration of corticosteroids and activated protein C. Septic shock is severe sepsis with organ hypoperfusion and hypotension that are poorly responsive to initial fluid resuscitation. Common causes of septic shock include hospital-acquired gram-negative bacilli or gram-positive cocci, and often occur in immunocompromised patients and patients with chronic and debilitating diseases. Diagnosis is clinical, and other causes of shock should be ruled out via history, physical examination, ECG, and serum cardiac markers. Monitoring includes CBC, ABGs, chest x-ray, serum electrolytes, lactate levels or sublingual PCO2, and liver function. At the onset of septic shock, the WBC

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input_2 = "What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

# Combination 1 (k = 2, temperature = 0.2, top_p = 0.8)
answer = generate_rag_response(
    user_input_2,
    k=2,
    temperature = 0.2,
    top_p = 0.8
)
print (answer)

Llama.generate: prefix-match hit


###Answer
The common symptoms of appendicitis include abdominal pain, anorexia, and abdominal tenderness. Appendicitis cannot be cured via medicine alone; the standard treatment is surgical removal of the appendix, which can be done through open or laparoscopic appendectomy.


In [71]:
user_input_2 = "What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

# Combination 2 (k = 3, temperature = 0.3, top_p = 0.85)
answer = generate_rag_response(
    user_input_2,
    k=3,
    temperature = 0.3,
    top_p = 0.85
)
print (answer)

Llama.generate: prefix-match hit


Based on the context provided, the answer would be:
The common symptoms of appendicitis include abdominal pain, anorexia, and abdominal tenderness. Appendicitis cannot be cured via medicine alone; instead, surgical removal of the appendix (appendectomy) is required for treatment. This procedure should be preceded by IV antibiotics.


In [81]:
user_input_2 = "What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

# Combination 3 (k = 4, temperature = 0.5, top_p = 0.9)
answer = generate_rag_response(
    user_input_2,
    k=4,
    temperature = 0.5,
    top_p = 0.9
)
print (answer)

Llama.generate: prefix-match hit


I don't know if I can definitively answer the second part of your question based on the context alone, as it doesn't explicitly state that appendicitis cannot be cured with medicine. However, according to the information provided, the standard treatment for appendicitis is surgical removal (appendectomy).

As for the first part of your question, the common symptoms of appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia; after a few hours, the pain shifts to the right lower quadrant. Pain increases with cough and motion. Classic signs are right lower quadrant direct and rebound tenderness located at McBurney's point (junction of the middle and outer thirds of the line joining the umbilicus to the anterior superior spine). Additional signs are pain felt in the right lower quadrant with palpation of the left lower quadrant (Rovsing sign), an increase in pain from passive extension of the right hip joint that stretches the iliopsoas muscle

In [78]:
user_input_2 = "What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

# Combination 4 (k = 5, temperature = 0.7, top_p = 0.95)
answer = generate_rag_response(
    user_input_2,
    k=5,
    temperature = 0.7,
    top_p = 0.95
)
print (answer)

Llama.generate: prefix-match hit


Common symptoms of appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia. After a few hours, the pain shifts to the right lower quadrant. Pain increases with cough and motion. Classic signs are right lower quadrant direct and rebound tenderness located at McBurney's point (junction of the middle and outer thirds of the line joining the umbilicus to the anterior superior spine). Additional signs include pain felt in the right lower quadrant with palpation of the left lower quadrant (Rovsing sign), an increase in pain from passive extension of the right hip joint that stretches the iliopsoas muscle (psoas sign), or pain caused by passive internal rotation of the flexed thigh (obturator sign). Low-grade fever is common.

Appendicitis cannot be cured via medicine alone, and surgery (open or laparoscopic appendectomy) is required for treatment. It's important to note that a negative appendectomy rate of 15% is considered acceptable due to the

In [79]:
user_input_2 = "What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

# Combination 5 (k = 5, temperature = 0.4, top_p = 0.92)
answer = generate_rag_response(
    user_input_2,
    k=5,
    temperature = 0.4,
    top_p = 0.92
)
print (answer)

Llama.generate: prefix-match hit


###Answer
The common symptoms of appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia; after a few hours, the pain shifts to the right lower quadrant. Pain increases with cough and motion. Classic signs are right lower quadrant direct and rebound tenderness located at McBurney's point. Other signs include pain felt in the right lower quadrant with palpation of the left lower quadrant (Rovsing sign), an increase in pain from passive extension of the right hip joint, or pain caused by passive internal rotation of the flexed thigh (obturator sign). A low-grade fever is common. However, these classic findings appear in less than 50% of patients. Many variations of symptoms and signs occur, and pain may not be localized, particularly in infants and children. Tenderness may be diffuse or absent, and bowel movements are usually less frequent or absent if diarrhea is a sign, a retrocecal appendix should be suspected.

Appendicitis cannot be cur

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

# Combination 1 (k = 2, temperature = 0.2, top_p = 0.8)
answer = generate_rag_response(
    user_input_3,
    k=2,
    temperature = 0.2,
    top_p = 0.8
)
print (answer)

Llama.generate: prefix-match hit


Based on the context provided, the condition being described is Alopecia Areata. The effective treatments for this condition include topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA). The possible cause behind this condition is an autoimmune disorder.


In [92]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

# Combination 2 (k = 3, temperature = 0.3, top_p = 0.85)
answer = generate_rag_response(
    user_input_3,
    k=3,
    temperature = 0.3,
    top_p = 0.85
)
print (answer)

Llama.generate: prefix-match hit


I don't know. The context mentions that sudden patchy hair loss is a symptom of alopecia areata, but it does not provide information about effective treatments or solutions for this condition. It only mentions various treatment options such as topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA). However, it does not specify which of these treatments are effective for addressing sudden patchy hair loss.


In [80]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

# Combination 3 (k = 4, temperature = 0.5, top_p = 0.9)
answer = generate_rag_response(
    user_input_3,
    k=4,
    temperature = 0.5,
    top_p = 0.9
)
print (answer)

Llama.generate: prefix-match hit


Based on the context provided, the condition being described is likely Alopecia Areata. According to the Merck Manual, the treatment for Alopecia Areata includes corticosteroids (topical or oral), topical anthralin, minoxidil, or systemic immunosuppressants in severe cases. The cause of Alopecia Areata is thought to be an autoimmune disorder affecting genetically susceptible people exposed to unclear environmental triggers.

Therefore, the effective treatments for addressing sudden patchy hair loss on the scalp could be corticosteroids (topical or oral), topical anthralin, minoxidil, or systemic immunosuppressants in severe cases. The possible causes behind this condition are believed to be autoimmune disorders and unclear environmental triggers.


In [82]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

# Combination 4 (k = 5, temperature = 0.7, top_p = 0.95)
answer = generate_rag_response(
    user_input_3,
    k=5,
    temperature = 0.7,
    top_p = 0.95
)
print (answer)

Llama.generate: prefix-match hit


Based on the context provided, the condition described in the question is likely Alopecia Areata. The treatments mentioned for this condition include topical corticosteroids, sometimes in combination with topical anthralin or minoxidil, oral corticosteroids, and immunotherapy using diphencyprone or squaric acid dibutylester. These treatments aim to reduce inflammation, promote hair regrowth, and suppress the immune system's attack on the hair follicles. The possible causes behind this condition include genetics and environmental triggers, such as stress, infections, or certain medications. It is important to note that alopecia areata may also spontaneously regress, become chronic, or spread diffusely, so ongoing treatment might be necessary for some individuals.


In [83]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

# Combination 5 (k = 5, temperature = 0.4, top_p = 0.92)
answer = generate_rag_response(
    user_input_3,
    k=5,
    temperature = 0.4,
    top_p = 0.92
)
print (answer)

Llama.generate: prefix-match hit


Based on the context provided, the condition being described is likely Alopecia Areata. The effective treatments for this condition include topical or intralesional corticosteroids, topical anthralin, minoxidil, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA). The cause of this condition is believed to be an autoimmune disorder affecting genetically susceptible individuals exposed to unclear environmental triggers.


### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"

# Combination 1 (k = 2, temperature = 0.2, top_p = 0.8)
answer = generate_rag_response(
    user_input_4,
    k=2,
    temperature = 0.2,
    top_p = 0.8
)
print (answer)

Llama.generate: prefix-match hit


###Answer
The treatment for a person with a physical injury to brain tissue that temporarily or permanently impairs brain function includes ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed in patients with more severe injuries to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas. In the first few days after the injury, maintaining adequate brain perfusion and oxygenation and preventing complications of altered sensorium are important. Subsequently, many patients require rehabilitation. There is no specific treatment for the brain injury itself, but supportive care should include preventing systemic complications due to immobilization, providing good nutrition, and preventing pressure ulcers.


In [91]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"

# Combination 2 (k = 3, temperature = 0.3, top_p = 0.85)
answer = generate_rag_response(
    user_input_4,
    k=3,
    temperature = 0.3,
    top_p = 0.85
)
print (answer)

Llama.generate: prefix-match hit


I. Supportive care: Preventing systemic complications due to immobilization (eg, pneumonia, UTI, thromboembolic disease), providing good nutrition, and preventing pressure ulcers.
II. Initial treatment: Ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed for patients with more severe injury to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas.
III. First few days after injury: Maintaining adequate brain perfusion and oxygenation and preventing complications of altered sensorium.
IV. Long-term treatment: Depending on the severity of the injury, some patients may require rehabilitation. For those with significant structural injury or a Glasgow Coma Scale score of < 10, drugs are given to prevent seizures during the first week after injury but should be stopped after 1 week unless seizures occur. If seizures begin > 1 week 

In [84]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"

# Combination 3 (k = 4, temperature = 0.5, top_p = 0.9)
answer = generate_rag_response(
    user_input_4,
    k=4,
    temperature = 0.5,
    top_p = 0.9
)
print (answer)

Llama.generate: prefix-match hit


I. Supportive care: There is no specific treatment for traumatic brain injury (TBI), but supportive care should include preventing systemic complications due to immobilization, providing good nutrition, and preventing pressure ulcers.
II. Initial treatment: Ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure are essential in the first few days after the injury. Surgery may be needed for patients with more severe injury to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas.
III. Preventing seizures: Drugs are given to prevent seizures in patients with significant structural injury (eg, large contusions or hematomas, brain laceration, depressed skull fracture) or a Glasgow Coma Scale score of < 10. These drugs reduce the risk of seizures during the first week after injury but do not prevent permanent posttraumatic epilepsy months or years later. 

In [85]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"

# Combination 4 (k = 5, temperature = 0.7, top_p = 0.95)
answer = generate_rag_response(
    user_input_4,
    k=5,
    temperature = 0.7,
    top_p = 0.95
)
print (answer)

Llama.generate: prefix-match hit


I. Supportive care: This includes preventing systemic complications due to immobilization, providing good nutrition, and preventing pressure ulcers. 
II. Seizure prevention: For those with significant structural injury or a Glasgow Coma Scale (GCS) score < 10, prophylactic anticonvulsants are considered to reduce the risk of seizures during the first week after injury. These drugs do not prevent permanent posttraumatic epilepsy months or years later and should be stopped after one week unless seizures occur then. If seizures begin more than one week after head injury, long-term treatment with drugs is required. 
III. No specific treatment: There is no cure for traumatic brain injuries, but supportive care is essential to maintain adequate brain perfusion and oxygenation and prevent complications of altered sensorium. In the first few days, maintaining adequate brain perfusion and oxygenation is crucial. Subsequently, many patients require rehabilitation. 
IV. Hypothermia: Therapeutic s

In [86]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"

# Combination 5 (k = 5, temperature = 0.4, top_p = 0.92)
answer = generate_rag_response(
    user_input_4,
    k=5,
    temperature = 0.4,
    top_p = 0.92
)
print (answer)

Llama.generate: prefix-match hit


Based on the context provided, the following treatments are recommended for a person with traumatic brain injury (TBI):
1. Ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure.
2. Surgery to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas.
3. Supportive care, including preventing systemic complications due to immobilization, providing good nutrition, and preventing pressure ulcers.
4. In the first few days after the injury, maintaining adequate brain perfusion and oxygenation and preventing complications of altered sensorium are important.
5. Subsequently, many patients require rehabilitation.
6. For patients with significant structural injury (eg, larger contusions or hematomas, brain laceration, depressed skull fracture) or a Glasgow Coma Scale (GCS) score < 10, a prophylactic anticonvulsant should be considered to prevent seizures.
7. If s

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"

# Combination 1 (k = 2, temperature = 0.2, top_p = 0.8)
answer = generate_rag_response(
    user_input_5,
    k=2,
    temperature = 0.2,
    top_p = 0.8
)
print (answer)

Llama.generate: prefix-match hit


Based on the context provided, here's the answer:

The person with a fractured leg should follow these steps:
1. Rest and elevate the injured limb above heart level for the first 2 days to minimize swelling.
2. Apply elastic bandages (layers 2 and 4) to the injured area.
3. Immobilize the joints proximal and distal to the injury using a cast or splint, depending on the severity and healing time.
4. Avoid putting objects inside the cast and keep it dry.
5. Inspect the skin around the cast daily for redness or soreness and apply lotion if needed.
6. Pad any rough edges of the cast to prevent irritation.
7. Apply warmth (heating pad) for 15 to 20 minutes after the first 48 hours to relieve pain and speed healing.
8. Seek medical care immediately if an odor emanates from within the cast or a fever develops, as these may indicate infection.
9. For fractures requiring prolonged immobilization (more than 3-4 weeks), consider early mobilization to minimize contractures and muscle atrophy.
10. 

In [90]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"

# Combination 2 (k = 3, temperature = 0.3, top_p = 0.85)
answer = generate_rag_response(
    user_input_5,
    k=3,
    temperature = 0.3,
    top_p = 0.85
)
print (answer)

Llama.generate: prefix-match hit


Based on the context provided, here's the answer:

The person with a fractured leg should follow these steps:
1. Rest and elevate the injured limb above heart level for the first 2 days to minimize swelling.
2. Apply ice after 48 hours for 15 to 20 minutes to relieve pain and speed healing.
3. Immobilize the injury using a cast or splint, depending on the severity and duration of immobilization required. If a cast is used, keep it dry, avoid putting objects inside it, inspect the skin around it daily, and seek medical care if there's an odor, fever, or signs of infection.
4. Periodically apply warmth using a heating pad for pain relief after 48 hours.
5. Inspect the edges of the cast and skin regularly for redness or soreness and apply lotion to these areas.
6. If immobilization lasts more than a few days, maintain good hygiene and prevent complications such as deep venous thrombosis, urinary tract infection, stiffness, contractures, and muscle atrophy.
7. For fractures that require pr

In [87]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"

# Combination 3 (k = 4, temperature = 0.5, top_p = 0.9)
answer = generate_rag_response(
    user_input_5,
    k=4,
    temperature = 0.5,
    top_p = 0.9
)
print (answer)

Llama.generate: prefix-match hit


Based on the context provided, here's the answer:

The person with a fractured leg should first receive supportive measures such as pain relief and local wound care. Immobilization is essential to prevent further injury and facilitate healing. This can be achieved through the use of a cast or splint depending on the severity and duration of immobilization required. The injured limb should be elevated above heart level for the first 2 days to minimize swelling and promote drainage of edema fluid. After 48 hours, periodic application of warmth may relieve pain and speed healing.

The cast or splint should be kept dry, and patients are instructed not to put any objects inside it. The edges and skin around the cast should be inspected daily for redness or soreness, and lotion applied if needed. Patients should also be advised to seek medical care immediately if an odor emanates from within the cast or a fever develops.

For pain relief, analgesics may be used, including opioids in severe c

In [88]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"

# Combination 4 (k = 5, temperature = 0.7, top_p = 0.95)
answer = generate_rag_response(
    user_input_5,
    k=5,
    temperature = 0.7,
    top_p = 0.95
)
print (answer)

Llama.generate: prefix-match hit


Based on the context provided, here's the answer:

The necessary precautions for a person with a fractured leg include immobilizing the injured limb using elastic bandages or a cast, elevating it above heart level for the first two days to minimize swelling and promote healing, and applying warmth after 48 hours to relieve pain. The person should also be instructed to keep the cast dry, not put objects inside it, inspect the skin around it daily, and seek medical care if they notice any signs of infection such as an odor from within the cast or a fever. A splint can be used for less severe injuries that require immobilization for several days, while a cast is typically used for fractures that need weeks of immobilization. The person should also avoid putting weight on the affected leg and keep it elevated as much as possible to prevent complications such as stiffness, contractures, and muscle atrophy. If prolonged bed rest is required for the fracture, precautions against deep venous t

In [89]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"

# Combination 5 (k = 5, temperature = 0.4, top_p = 0.92)
answer = generate_rag_response(
    user_input_5,
    k=5,
    temperature = 0.4,
    top_p = 0.92
)
print (answer)

Llama.generate: prefix-match hit


Based on the context provided, here is the answer:

The person with a fractured leg should first receive supportive measures such as immobilization using either a cast or splint. The injured limb should be elevated above heart level for the first 2 days to minimize swelling and help drain edema fluid. After 48 hours, periodic application of warmth for 15 to 20 minutes may relieve pain and speed healing. Immobilization decreases pain and facilitates healing by preventing further injury. Joints proximal and distal to the injury should also be immobilized.

The person with a fractured leg should keep their cast dry, never put an object inside it, inspect the cast's edges and skin around it every day, apply lotion to any red or sore areas, and pad any rough edges with soft adhesive tape, cloth, or other soft material to prevent the cast's edges from causing discomfort. They should also seek medical care at once if an odor emanates from within the cast or if a fever develops, as these may i

**Observations:**

`retriever parameter k`
*   Increasing the retriever parameter k upto a certain point improved the contextual richness and gave clinically detailed responses.
*   **Lower to moderate values of k** resulted in more **concise information**.
*   **Higher value of k** resulted in a **more comprehensive and grounded response** but also included **irrelevant information sometimes**.

`top_p`
*   **Lower to moderate values of top_p** resulted in **concise, focussed and complete responses**.
*   **Higher value of top_p (0.95)** gave **more detailed responses** but also **increased verbosity**.

`temperature`
*   **Lower temperature** values gave more **focused and contextually grounded** responses.
*   **Higher temperature** values generated **elaborated responses with additional supporting information** from context but this increases the risk of possibly **less grounded** and **model interpreted responses** rather than directly coming from context.


## **Output Evaluation**

- The LLM-as-a-judge method is used to check the quality of the RAG system on two parameters - retrieval and generation.

- The same Mistral model is used for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [51]:
# Prompt to evaluate how well the answer is grounded in the provided context
groundedness_rater_system_message  = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
The answer should be derived only from the information presented in the context

Instructions:
1. First write down the steps that are needed to evaluate the answer as per the metric.
2. Give a step-by-step explanation if the answer adheres to the metric considering the question and context as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the answer using the evaluaton criteria and assign a score.
"""

In [52]:
## Prompt to evaluate how relevant the answer is to the original question
relevance_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
Relevance measures how well the answer addresses the main aspects of the question, based on the context.
Consider whether all and only the important aspects are contained in the answer when evaluating relevance.

Instructions:
1. First write down the steps that are needed to evaluate the context as per the metric.
2. Give a step-by-step explanation if the context adheres to the metric considering the question as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the context using the evaluaton criteria and assign a score.
"""

In [53]:
# Template for providing evaluation input
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [39]:
def generate_ground_relevance_response(user_input,k=3,max_tokens=512,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=3)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    answer =  response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [54]:
user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"
ground,rel = generate_ground_relevance_response(user_input_1)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the key information related to managing sepsis in a critical care unit from the context.
2. Compare the AI generated answer with the identified key information.
3. Determine if the answer is derived only from the information presented in the context.

Explanation:
The AI generated answer includes several steps for managing sepsis in a critical care unit, such as giving parenteral antibiotics after taking specimens for Gram stain and culture, starting very prompt empiric therapy, changing the antibiotic regimen based on culture and sensitivity results, treating septic shock with different antibiotics, draining abscesses and excising necrotic tissues, normalizing blood glucose, using corticosteroid therapy, and considering other emerging therapies.

The context provides detailed information about the critical care unit, patient monitoring and testing, blood tests, and managing sepsis specifically. The key information related to managing sepsis i

**Observations:**
*   **Groundedness**
    *   The entire generated response contained detailed response for sepsis management sourced directly context.
    *   Based on the evaluation criteria, the rating is '**5: The metric is followed completely'**.
*   **Relevance**
    *   The context covered all the important aspects of the question without giving any irrelavant details.
    *   The rating is '**4: The metric is followed mostly'**.    


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [55]:
user_input_2 = "What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
ground,rel = generate_ground_relevance_response(user_input_2)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the information in the context related to the question.
2. Determine if the symptoms mentioned in the question are present in the context.
3. Check if the treatment options given in the answer are consistent with the context.
4. Verify that the answer is derived solely from the context and not any external knowledge.

Explanation:
The question asks about the common symptoms of appendicitis and whether it can be cured via medicine. The context provides information on the diagnosis, treatment, and prognosis of appendicitis. The symptoms mentioned in the question are present in the context as abdominal pain, anorexia, and abdominal tenderness. The context also states that surgery is the standard treatment for appendicitis and that antibiotics alone cannot cure it. Therefore, the answer "The common symptoms of appendicitis include abdominal pain, anorexia, and abdominal tenderness. Appendicitis cannot be cured via medicine alone; the standard trea

**Observations:**
*   **Groundedness**
    *   The generated response correctly identified the symptoms of appendicitis and the standard treatment from the context.
    *   The extent to which the metric is followed is '**5: The metric is followed completely'**.
*   **Relevance**
    *   The retrieved documents specifically addressed the symptoms of appendicitis and standard surgical treatment and did not include any irrelevant information.
    *   The extent to which the metric is followed is '**5: The metric is followed completely**'.     

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [56]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
ground,rel = generate_ground_relevance_response(user_input_3)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the information in the context related to sudden patchy hair loss and its treatments.
2. Determine if the AI generated answer includes only the information from the context.
3. Check if the possible causes mentioned in the answer are also present in the context.

The answer adheres to the metric as it mentions the treatment options for sudden patchy hair loss (alopecia areata) which are directly derived from the context. The possible causes of alopecia areata are also mentioned in the context and included in the answer. Therefore, the answer is a complete representation of the information present in the context.

Rating: 5 - The metric is followed completely.

 Steps to evaluate the context as per the relevance metric:
1. Identify the main aspects of the question: effective treatments or solutions for sudden patchy hair loss and possible causes.
2. Determine if the context provides information on these main aspects.
3. Check if all important a

**Observations:**
*   **Groundedness**
    *   The generated response mentions the causes and treatment options for sudden patchy hair loss (alopecia areata) which are directly retrieved from context.
    *   The rating is '**5: The metric is followed completely'**.
*   **Relevance**
    *   The retrieved documents specifically mentioned the causes and treatment options for sudden patchy hair loss without any irrelevant information.
    *   The extent to which the metric is followed is '**4: The metric is followed mostly**'.
    

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [57]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
ground,rel = generate_ground_relevance_response(user_input_4)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the information in the context related to treatments for traumatic brain injury (TBI).
2. Compare the AI generated answer with the identified information from the context.
3. Determine if the answer is derived only from the information presented in the context.

Explanation:
The AI generated answer mentions several treatments for TBI, including ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure, surgery to place monitors or decompress the brain, rehabilitation, preventing systemic complications due to immobilization, providing good nutrition, preventing pressure ulcers, and drugs to prevent seizures.

The context provides information on the initial treatment for TBI, which includes ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. It also mentions surgery for more severe cases and rehabilitation for many patients. The context does not mention specific

**Observations:**
*   **Groundedness**
    *   The generated answer includes several treatment options for traumatic brain injury discussed in detail.
    *   The answer is directly derived from the context, hence the rating is '**5: The metric is followed completely'**.
*   **Relevance**
    *   The context specifically focuses on the treatment of brain injuries and does not include any irrelevant information.
    *   The extent to which the metric is followed is '**4: The metric is followed mostly**'.     

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [58]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
ground,rel = generate_ground_relevance_response(user_input_5)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the information in the context related to precautions and treatment for a fractured leg.
2. Determine if the AI generated answer includes all the necessary precautions and treatment steps mentioned in the context.
3. Check if the AI generated answer is derived only from the information presented in the context.

Explanation:
The AI generated answer includes all the necessary precautions and treatment steps for a person with a fractured leg as mentioned in the context. The answer states that the injured limb should be immobilized using elastic bandages or a cast, elevated above heart level for the first two days to minimize swelling, and application of warmth for pain relief after 48 hours. It also mentions avoiding putting objects inside the cast, keeping the cast dry, inspecting the skin around the cast daily, seeking medical care if an odor is detected or a fever develops, and considering bivalving the cast if severe swelling is expected.

F

**Observations:**
*   **Groundedness**
    *   The entire generated response for leg fracture included detailed precautions, treatment and recovery steps sourced directly from the retrieved documents.
    *   The rating is '**5: The metric is followed completely'**.
*   **Relevance**
    *   The context provides sufficient details for each part of the question.
    *   The extent to which the metric is followed is '**5: The metric is followed completely**'.     

## Actionable Insights and Business Recommendations

**Observations and Insights:**
*   The number of pages in the pdf document significantly increase the time taken for vector database creation.
*   Retrieval parameter k, LLM parameters, max_token and temperature, clearly influence the generated responses.
*   RAG-based medical assistant using Merck manual showed a significant improvement over standalone LLM responses.

**Business Recommendations:**
*   Deploying RAG-based medical assistants in healthcare would enable faster and more accurate diagnosis and treatments.
*   Low temperature settings are recommended in AI assistants for healthcare to minimize hallucinations.
*   Vector databases should be continuously updated with new medical literature.
*   For critical medical cases, human expert should be employed to review the responses.





<font size=6 color='blue'>Power Ahead</font>
___